# ISIC 2017 — 12-Metric Vertical Slice

Evaluates **7 FAE methods × 12 metrics** across **2 models** on **12 ISIC 2017 test images**.
Expected output: `results/vertical_slice_7fae_12metrics.csv` with 2016 rows
(2 models × 7 FAE × 12 images × 12 metrics).

**Prerequisites**
- Runtime: **T4 GPU** — Runtime → Change runtime type → T4 GPU
- Google Drive must contain:
  - `thesis/weights/resnet18_isic2017.pth`
  - `thesis/weights/squeezenet_isic2017.pth`

  Both are produced by `notebooks/colab_train.ipynb`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os
os.makedirs(f'{DRIVE_THESIS}/results', exist_ok=True)
print(f'Drive mounted. Thesis folder: {DRIVE_THESIS}')

## 2. Clone Repository

In [ ]:
import os
REPO_DIR = '/content/fae-metrics-master-thesis'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/dawkopagh/fae-metrics-master-thesis.git {REPO_DIR}
    %cd {REPO_DIR}

print(f'Working directory: {os.getcwd()}')
print('Commit:', end=' ')
!git rev-parse HEAD

## 3. Install Dependencies

Pins `quantus==0.6.0` explicitly until `requirements.txt` carries version pins.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0

import captum, quantus, torch
print(f'captum  {captum.__version__}')
print(f'quantus {quantus.__version__}')
print(f'torch   {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Copy Weights from Drive

Copies the production weights (trained 2026-04-22 on full ISIC 2017)
from Drive. SHA-256 values are from `weights/README.md`.

In [ ]:
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'

import os
os.makedirs('weights', exist_ok=True)

!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth    weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth  weights/squeezenet_isic2017.pth

print('SHA-256 checksums (actual):')
!sha256sum weights/resnet18_isic2017.pth weights/squeezenet_isic2017.pth
print(f'\nExpected resnet18:    {RESNET_SHA}')
print(f'Expected squeezenet:  {SQUEEZENET_SHA}')

## 5. Download ISIC 2017 Data

Downloads all three splits (train / val / test) to `data/`.
First run: ~6 GB, ~20-30 min on T4. `skip_existing=True` makes re-runs fast.

> **Path note**: `vertical_slice.py` is hard-coded to `data_root='data'`, so data
> must live at `data/images/test/…` — **not** `data/isic2017/images/test/…`.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.download_isic import download_isic2017

DATA_ROOT = 'data'
download_isic2017(dest_dir=DATA_ROOT, skip_existing=True)

import os
print('\nTest split verification:')
for cls in ['melanoma', 'nevus', 'seborrheic_keratosis']:
    img_path = f'{DATA_ROOT}/images/test/{cls}'
    msk_path = f'{DATA_ROOT}/masks/test/{cls}'
    n_img = len([f for f in os.listdir(img_path) if not f.startswith('.')]) if os.path.exists(img_path) else 0
    n_msk = len([f for f in os.listdir(msk_path) if not f.startswith('.')]) if os.path.exists(msk_path) else 0
    print(f'  {cls:28s}  images={n_img:4d}  masks={n_msk:4d}')

## 6. Profile Sub-Run

Runs 3 FAE methods × 12 metrics on 1 image to measure per-metric cost on the T4.
Projects total runtime for the full 7-FAE × 12-image slice before committing to it.

In [ ]:
import os
os.makedirs('results', exist_ok=True)

!python experiments/profile_metrics.py

import pandas as pd, shutil

# profile_metrics.py writes to results/profile_one_image.csv
profile_df = pd.read_csv('results/profile_one_image.csv')
shutil.copy('results/profile_one_image.csv', 'results/profile_t4.csv')

print('\n=== Top-10 slowest (fae_method, metric) ===')
top10 = profile_df.sort_values('seconds', ascending=False).head(10)
print(top10[['fae_method', 'metric', 'seconds']].to_string(index=False))

# Profile covers 3 FAE × 1 image; full run is 7 FAE × 12 images
profile_total_sec = profile_df['seconds'].sum()
projected_sec = profile_total_sec * 7 * 12 / 3
projected_min = projected_sec / 60

print(f'\nProfiled total:  {profile_total_sec:.1f} s  (3 FAE × 1 image)')
print(f'Projected 12-image full slice runtime: {projected_min:.1f} minutes')

if projected_sec > 45 * 60:
    print('\nWARNING: projected runtime exceeds 45 minutes.')
    print('Consider reducing ModelParameterRandomisation step count')
    print('or running on Colab Pro for longer session.')
else:
    print('\nProjected runtime acceptable — proceeding.')

## 7. Full 12-Metric Vertical Slice

2 models × 7 FAE × 12 images × 12 metrics = **2016 rows**.
Attribution maps are cached to `attributions_cache/` — the first run is slowest;
re-runs recompute only the metric scores.

In [ ]:
!python experiments/vertical_slice.py \
    --output-csv results/vertical_slice_7fae_12metrics.csv \
    --max-images 12

import pandas as pd

df = pd.read_csv('results/vertical_slice_7fae_12metrics.csv')
assert len(df) == 2016, f"Expected 2016 rows, got {len(df)}"
print(f'Row count: {len(df)} \u2713')
print(f'Columns:   {list(df.columns)}')

print('\nNaN counts per (fae_method, metric) — non-zero only:')
nan_counts = df.groupby(['fae_method', 'metric'])['score'].apply(lambda s: s.isna().sum())
nan_nonzero = nan_counts[nan_counts > 0]
if nan_nonzero.empty:
    print('  (none)')
else:
    print(nan_nonzero.to_string())
# Expected non-zero NaN:
#   completeness:     96 (4 non-axiomatic FAE × 2 models × 12 images)
#   non_sensitivity: 168 (7 FAE × 2 models × 12 images, disabled by default)

## 8. Aggregate and Rank

Applies Second Moment Scaling normalisation (Decision D1) and weighted-mean
aggregation (Decision D2) from `docs/thesis_plan.md`. NaN entries (Completeness
and NonSensitivity) are skipped automatically; remaining weights are renormalised.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from src.aggregation.normalize import normalize_scores
from src.aggregation.aggregate import aggregate_scores

df = pd.read_csv('results/vertical_slice_7fae_12metrics.csv')

normed = normalize_scores(df, method='second_moment')
ranked = aggregate_scores(normed, operator='weighted_mean')
ranked = ranked.sort_values('effectiveness_index', ascending=False)

ranked.to_csv('results/preliminary_ranking_12metrics.csv', index=False)
print(ranked.to_string(index=False))

## 9. Copy Results to Drive

In [ ]:
import shutil, os

results_to_save = [
    'results/profile_t4.csv',
    'results/vertical_slice_7fae_12metrics.csv',
    'results/preliminary_ranking_12metrics.csv',
]

for src in results_to_save:
    dest = f'{DRIVE_THESIS}/{src}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(src, dest)
    print(f'Saved {src} \u2192 {dest}')

print('\nAll results saved to Google Drive.')
print('Download these to fae-metrics-master-thesis/results/ locally before the next session.')